# B1 — Weak-Label Training Data Preparation

## Overview

This notebook prepares training and validation datasets for LoRA fine-tuning as part of the Heard project, a generative AI tool for analyzing employee feedback.

Using the processed Glassdoor review data provided by Part C, we examine aspect-level sentiment labels, perform stratified sampling, and convert the selected reviews into instruction-formatted JSONL files.

The notebook includes the following steps:

1. Data inspection and distribution analysis
2. Stratified sampling for training data
3. Instruction formatting and JSONL export
4. Validation dataset preparation
5. Data quality checks

Outputs:

* train_instructions.jsonl: 18,000 training samples
* validation_instructions.jsonl: 1,800 validation samples

The sentiment labels are weak labels derived from aspect ratings. These datasets will be used for aspect-conditioned sentiment classification during LoRA fine-tuning in B2.

## 1. Data Inspection

Import Pandas and NumPy for data processing and analysis. Configure Pandas display settings to make the dataset easier to inspect.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

In [2]:
train_path = "../data/processed/kaggle_aspect_train.csv"

df = pd.read_csv(train_path, nrows=10)

display(df)

,source,firm,date_review,job_title,location,review_text,aspect,aspect_rating,aspect_sentiment,task_text
0,kaggle_glassdoor,CBRE,2017-09-27 00:00:00+00:00,Assistant Facilities Manager,"Atlanta, GA",TITLE: Assistant Facilities Manager PROS: Great co-workers and forum to help the community and those in need. CONS: Career Advancement process nee...,work_life_balance,4.0,positive,ASPECT: work_life_balance REVIEW: TITLE: Assistant Facilities Manager PROS: Great co-workers and forum to help the community and those in need. CO...
1,kaggle_glassdoor,Salesforce,2021-02-11 00:00:00+00:00,Business Development Representative,"London, England, England",TITLE: Great place to work PROS: * hands-on mentoring * great virtual onboarding * career vision * great career progression * good work-life balan...,work_life_balance,5.0,positive,ASPECT: work_life_balance REVIEW: TITLE: Great place to work PROS: * hands-on mentoring * great virtual onboarding * career vision * great career ...
2,kaggle_glassdoor,Best-Western,2014-06-14 00:00:00+00:00,Front Desk,"Juneau, AK","TITLE: Great temporary opportunity! PROS: Great benefits, and amazing management make this place. I love overtime potential and there was plenty o...",work_life_balance,2.0,negative,"ASPECT: work_life_balance REVIEW: TITLE: Great temporary opportunity! PROS: Great benefits, and amazing management make this place. I love overtim..."
3,kaggle_glassdoor,Deloitte,2020-09-21 00:00:00+00:00,Data Engineer,Mumbai,PROS: Amazing work environment and helpful co-workers CONS: Not Applicable . Overall good experience,work_life_balance,4.0,positive,ASPECT: work_life_balance REVIEW: PROS: Amazing work environment and helpful co-workers CONS: Not Applicable . Overall good experience
4,kaggle_glassdoor,Citi,2021-05-04 00:00:00+00:00,Citiphones Officer,Chennai,TITLE: Good Salary PROS: They provide good salary for freshers and experienced. You also get allowance and other benefits CONS: Slow career growth...,work_life_balance,3.0,neutral,ASPECT: work_life_balance REVIEW: TITLE: Good Salary PROS: They provide good salary for freshers and experienced. You also get allowance and other...
5,kaggle_glassdoor,Microsoft,2013-04-29 00:00:00+00:00,Project Manager III,"Redmond, WA","TITLE: Good working conditions, lack of vision from the top PROS: Great benefits, professionalism, smart people CONS: bureaucratic, bean counters ...",work_life_balance,4.0,positive,"ASPECT: work_life_balance REVIEW: TITLE: Good working conditions, lack of vision from the top PROS: Great benefits, professionalism, smart people ..."
6,kaggle_glassdoor,Deloitte,2017-04-27 00:00:00+00:00,Senior Consultant,"Atlanta, GA",TITLE: Good Company PROS: Good Benefits Generous PTO Leadership supportive of professional development opportunities. CONS: Utilization requiremen...,work_life_balance,5.0,positive,ASPECT: work_life_balance REVIEW: TITLE: Good Company PROS: Good Benefits Generous PTO Leadership supportive of professional development opportuni...
7,kaggle_glassdoor,McDonald-s,2019-10-31 00:00:00+00:00,Crew Member,NaN,"TITLE: A good first job, only stay here for a maximum 12 months. Anything longer is an actual drag PROS: Very very flexible schedule Sometimes can...",work_life_balance,5.0,positive,"ASPECT: work_life_balance REVIEW: TITLE: A good first job, only stay here for a maximum 12 months. Anything longer is an actual drag PROS: Very ve..."
8,kaggle_glassdoor,PwC,2021-02-21 00:00:00+00:00,Manager,"New York, NY",TITLE: Company culture and opportunities PROS: Good company culture and opportunities to get involved outside of work hours. Can work with top com...,work_life_balance,2.0,negative,ASPECT: work_life_balance REVIEW: TITLE: Company culture and opportunities PROS: Good company culture and opportunities to get involved outside of...
9,kaggle_glassdoor,IBM,2009-10-01 00:00:00+00:00,Category Manager,NaN,"TITLE: IBM Category Manager, Integrated Supply Chain (ISC) Procurement PROS: Currently pay and benefits are better than most other com

In [3]:
print(df.columns.tolist())

['source', 'firm', 'date_review', 'job_title', 'location', 'review_text', 'aspect', 'aspect_rating', 'aspect_sentiment', 'task_text']


In [4]:
display(
    df[
        [
            "review_text",
            "aspect",
            "aspect_rating",
            "aspect_sentiment",
            "task_text"
        ]
    ].head()
)

,review_text,aspect,aspect_rating,aspect_sentiment,task_text
0,TITLE: Assistant Facilities Manager PROS: Great co-workers and forum to help the community and those in need. CONS: Career Advancement process nee...,work_life_balance,4.0,positive,ASPECT: work_life_balance REVIEW: TITLE: Assistant Facilities Manager PROS: Great co-workers and forum to help the community and those in need. CO...
1,TITLE: Great place to work PROS: * hands-on mentoring * great virtual onboarding * career vision * great career progression * good work-life balan...,work_life_balance,5.0,positive,ASPECT: work_life_balance REVIEW: TITLE: Great place to work PROS: * hands-on mentoring * great virtual onboarding * career vision * great career ...
2,"TITLE: Great temporary opportunity! PROS: Great benefits, and amazing management make this place. I love overtime potential and there was plenty o...",work_life_balance,2.0,negative,"ASPECT: work_life_balance REVIEW: TITLE: Great temporary opportunity! PROS: Great benefits, and amazing management make this place. I love overtim..."
3,PROS: Amazing work environment and helpful co-workers CONS: Not Applicable . Overall good experience,work_life_balance,4.0,positive,ASPECT: work_life_balance REVIEW: PROS: Amazing work environment and helpful co-workers CONS: Not Applicable . Overall good experience
4,TITLE: Good Salary PROS: They provide good salary for freshers and experienced. You also get allowance and other benefits CONS: Slow career growth...,work_life_balance,3.0,neutral,ASPECT: work_life_balance REVIEW: TITLE: Good Salary PROS: They provide good salary for freshers and experienced. You also get allowance and other...


## 2. Data Distribution Analysis

Load the first 20,000 rows for a preliminary analysis of aspect and sentiment distributions.

In [5]:
df = pd.read_csv(
    train_path,
    nrows=20000
)

print("Sample shape:", df.shape)

Sample shape: (20000, 10)


In [6]:
print("Aspect Distribution:")

display(
    df["aspect"].value_counts()
)

Aspect Distribution:


aspect
work_life_balance    20000
Name: count, dtype: int64

In [7]:
print("Sentiment Distribution:")

display(
    df["aspect_sentiment"].value_counts()
)

Sentiment Distribution:


aspect_sentiment
positive    10005
negative     5084
neutral      4911
Name: count, dtype: int64

In [8]:
aspect_sentiment_dist = pd.crosstab(
    df["aspect"],
    df["aspect_sentiment"]
)

display(aspect_sentiment_dist)

aspect_sentiment,negative,neutral,positive
aspect,,,
work_life_balance,5084,4911,10005


In [9]:
# Count aspect and sentiment distributions across the full dataset
aspect_counts = pd.Series(dtype="int64")
sentiment_counts = pd.Series(dtype="int64")

for chunk in pd.read_csv(
    train_path,
    usecols=["aspect", "aspect_sentiment"],
    chunksize=50000
):

    aspect_counts = aspect_counts.add(
        chunk["aspect"].value_counts(),
        fill_value=0
    )

    sentiment_counts = sentiment_counts.add(
        chunk["aspect_sentiment"].value_counts(),
        fill_value=0
    )

print("Full Aspect Distribution:")
display(aspect_counts.sort_values(ascending=False).astype(int))

print("Full Sentiment Distribution:")
display(sentiment_counts.sort_values(ascending=False).astype(int))

Full Aspect Distribution:


aspect
career_opp             529639
work_life_balance      527824
comp_benefits          527649
senior_mgmt            523194
culture_values         496367
diversity_inclusion    104453
dtype: int64

Full Sentiment Distribution:


aspect_sentiment
positive    1408125
neutral      652636
negative     648365
dtype: int64

Calculate the sentiment distribution for each aspect across the complete training dataset. These results will guide the stratified sampling strategy.

In [ ]:
aspect_sentiment_counts = pd.DataFrame()

for chunk in pd.read_csv(
    train_path,
    usecols=["aspect", "aspect_sentiment"],
    chunksize=50000
):

    chunk_counts = pd.crosstab(
        chunk["aspect"],
        chunk["aspect_sentiment"]
    )

    aspect_sentiment_counts = aspect_sentiment_counts.add(
        chunk_counts,
        fill_value=0
    )

display(aspect_sentiment_counts.fillna(0).astype(int))

aspect_sentiment,negative,neutral,positive
aspect,,,
career_opp,119537,129007,281095
comp_benefits,118803,144839,264007
culture_values,104947,100294,291126
diversity_inclusion,13041,17354,74058
senior_mgmt,158308,131313,233573
work_life_balance,133729,129829,264266


## 3. Stratified Sampling

Set the target sample size to 1,000 records per aspect-sentiment combination and define a random seed for reproducibility.

With six aspects and three sentiment classes, the target training dataset contains 18,000 samples.

In [11]:
# Sampling configuration

SAMPLES_PER_GROUP = 1000
RANDOM_STATE = 42

sampled_chunks = []

In [12]:
# Random sampling from the full training dataset
rng = np.random.default_rng(RANDOM_STATE)

sampled_chunks = []

for chunk in pd.read_csv(
    train_path,
    chunksize=50000
):

    chunk = chunk.dropna(
        subset=["review_text", "aspect", "aspect_sentiment"]
    )

    chunk = chunk.copy()
    chunk["_random"] = rng.random(len(chunk))

    # Keep aspect and sentiment columns
    chunk_sample = (
        chunk
        .sort_values("_random")
        .groupby(
            ["aspect", "aspect_sentiment"],
            group_keys=False
        )
        .head(SAMPLES_PER_GROUP)
    )

    sampled_chunks.append(chunk_sample)

print("Number of processed chunks:", len(sampled_chunks))

Number of processed chunks: 55


In [13]:
# Combine candidate samples

candidates = pd.concat(
    sampled_chunks,
    ignore_index=True
)

print("Candidate shape:", candidates.shape)

Candidate shape: (180000, 11)


Sort candidates by random priority, remove duplicate review-aspect combinations, and select up to 1,000 records per aspect-sentiment group.

The resulting dataset contains 18,000 balanced training samples.

In [14]:
train_sample = (
    candidates
    .sort_values("_random")
    .drop_duplicates(subset=["review_text", "aspect"])
    .groupby(
        ["aspect", "aspect_sentiment"],
        group_keys=False
    )
    .head(SAMPLES_PER_GROUP)
    .drop(columns=["_random"])
    .reset_index(drop=True)
)

print("Final training sample shape:", train_sample.shape)

Final training sample shape: (18000, 10)


Check the final aspect-sentiment distribution, total sample count, and number of unique reviews to confirm that the sampling targets have been met.

In [15]:
sample_distribution = pd.crosstab(
    train_sample["aspect"],
    train_sample["aspect_sentiment"]
)

display(sample_distribution)

print("Total samples:", len(train_sample))

print(
    "Unique review texts:",
    train_sample["review_text"].nunique()
)

aspect_sentiment,negative,neutral,positive
aspect,,,
career_opp,1000,1000,1000
comp_benefits,1000,1000,1000
culture_values,1000,1000,1000
diversity_inclusion,1000,1000,1000
senior_mgmt,1000,1000,1000
work_life_balance,1000,1000,1000


Total samples: 18000
Unique review texts: 17717


## 4. Instruction Formatting

Define an instruction for aspect-conditioned sentiment classification.

Convert each training record into an instruction-input-output structure, where the model receives an aspect and employee review and learns to generate a sentiment label in JSON format.

In [ ]:
import json

instruction = (
    "Analyze the employee review and determine the sentiment "
    "for the specified aspect. "
    "Choose one of: positive, neutral, negative. "
    "Return only a JSON object."
)

# Convert each row into an instruction-following example
def create_instruction_sample(row):
    
    input_text = (
        f"Aspect: {row['aspect']}\n"
        f"Review: {row['review_text']}"
    )

    output_text = json.dumps({
        "sentiment": row["aspect_sentiment"]
    })

    return {
        "instruction": instruction,
        "input": input_text,
        "output": output_text
    }

# Create training examples
training_data = [
    create_instruction_sample(row)
    for _, row in train_sample.iterrows()
]

print("Number of training examples:", len(training_data))

Number of training examples: 18000


In [21]:
print("First training example:")

print(
    json.dumps(
        training_data[0],
        indent=2,
        ensure_ascii=False
    )
)

First training example:
{
  "instruction": "Analyze the employee review and determine the sentiment for the specified aspect. Choose one of: positive, neutral, negative. Return only a JSON object.",
  "input": "Aspect: culture_values\nReview: TITLE: great job PROS: team collaboration, fun working environment, good opportunities CONS: can be a bit dangerous depending on the area you're working in",
  "output": "{\"sentiment\": \"positive\"}"
}


In [22]:
from pathlib import Path

# Create output directory
output_dir = Path("../part_b/data")
output_dir.mkdir(parents=True, exist_ok=True)

# Save training data
output_path = output_dir / "train_instructions.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for sample in training_data:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print("Saved to:", output_path)
print("Total examples:", len(training_data))

Saved to: ../part_b/data/train_instructions.jsonl
Total examples: 18000


In [23]:
# Read the first 3 saved examples

with open(output_path, "r", encoding="utf-8") as f:
    for i in range(3):
        line = f.readline()
        sample = json.loads(line)

        print(f"\nExample {i + 1}:")
        print(json.dumps(sample, indent=2, ensure_ascii=False))


Example 1:
{
  "instruction": "Analyze the employee review and determine the sentiment for the specified aspect. Choose one of: positive, neutral, negative. Return only a JSON object.",
  "input": "Aspect: culture_values\nReview: TITLE: great job PROS: team collaboration, fun working environment, good opportunities CONS: can be a bit dangerous depending on the area you're working in",
  "output": "{\"sentiment\": \"positive\"}"
}

Example 2:
{
  "instruction": "Analyze the employee review and determine the sentiment for the specified aspect. Choose one of: positive, neutral, negative. Return only a JSON object.",
  "input": "Aspect: work_life_balance\nReview: TITLE: It was awesome! PROS: Excellent benefits: free food, massages, great campus and awesome team mates. Great pay and work ethic. The trips overseas are fun too. CONS: Growth is pretty hard. Managers tend to micro manage and that is one of the few reasons I had left.",
  "output": "{\"sentiment\": \"positive\"}"
}

Example 3:


## 5. Validation Dataset Preparation
Specify the path to the validation dataset provided by Part C.

The validation data is kept separate from the training data to support independent model validation.

In [24]:
validation_path = "../data/processed/kaggle_aspect_validation.csv"

print("Validation path:", validation_path)

Validation path: ../data/processed/kaggle_aspect_validation.csv


In [25]:
validation_df = pd.read_csv(
    validation_path,
    usecols=[
        "review_text",
        "aspect",
        "aspect_sentiment"
    ]
)

print("Validation shape:", validation_df.shape)

display(validation_df.head())

Validation shape: (338400, 3)


,review_text,aspect,aspect_sentiment
0,TITLE: Great place to start in Financial Services PROS: -Ample amount of work and experiences for those eager enough -Brand name firm for resume -...,work_life_balance,positive
1,TITLE: great company to work PROS: work life balance is great CONS: salary is below Seattle area average,work_life_balance,positive
2,"TITLE: Overall, a valuable career experience. PROS: Great experience at a fast pace. Tons of resume-building opportunities, leadership roles, and ...",work_life_balance,neutral
3,"TITLE: Management consultant PROS: Nice projects,lots of data analytics CONS: Lack of client interaction as it's back-end",work_life_balance,positive
4,"TITLE: Great environment to work in! PROS: Great internal mobility should you wish to pursue it, the benefits are also great! You are constantly s...",work_life_balance,positive


In [26]:
validation_distribution = pd.crosstab(
    validation_df["aspect"],
    validation_df["aspect_sentiment"]
)

display(validation_distribution)

aspect_sentiment,negative,neutral,positive
aspect,,,
career_opp,14981,15976,35164
comp_benefits,14846,17984,33073
culture_values,13177,12487,36415
diversity_inclusion,1657,2175,9185
senior_mgmt,19779,16378,29220
work_life_balance,16703,16006,33194


In [27]:
# Validation sampling

VALIDATION_PER_GROUP = 100

validation_sample = (
    validation_df
    .dropna(
        subset=["review_text", "aspect", "aspect_sentiment"]
    )
    .groupby(
        ["aspect", "aspect_sentiment"],
        group_keys=False
    )
    .sample(
        n=VALIDATION_PER_GROUP,
        random_state=42
    )
    .reset_index(drop=True)
)

print("Validation sample shape:", validation_sample.shape)

Validation sample shape: (1800, 3)


In [28]:
validation_data = [
    create_instruction_sample(row)
    for _, row in validation_sample.iterrows()
]

print("Number of validation examples:", len(validation_data))

print(
    json.dumps(
        validation_data[0],
        indent=2,
        ensure_ascii=False
    )
)

Number of validation examples: 1800
{
  "instruction": "Analyze the employee review and determine the sentiment for the specified aspect. Choose one of: positive, neutral, negative. Return only a JSON object.",
  "input": "Aspect: career_opp\nReview: TITLE: Represents everything Research should not invited lnvolve PROS: If you want to know how research/ science should not be conducted join this university. CONS: I had my work stolen by my supervisor, he presented it as his. The only reason explanation is blatant racism. Whilst doing my PHD I was also subjected to racial comments and witnessed many first hand. NI has a long way to catch up to the rest of the U.K., including its universities",
  "output": "{\"sentiment\": \"negative\"}"
}


In [29]:
validation_output_path = output_dir / "validation_instructions.jsonl"

with open(validation_output_path, "w", encoding="utf-8") as f:
    for sample in validation_data:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print("Saved to:", validation_output_path)
print("Total examples:", len(validation_data))

Saved to: ../part_b/data/validation_instructions.jsonl
Total examples: 1800


## 6. Data Quality Checks
Compare unique review texts between the training and validation samples to detect exact duplicates across the two datasets.

No overlapping review texts were found.

In [ ]:
train_reviews = set(
    train_sample["review_text"].dropna()
)

validation_reviews = set(
    validation_sample["review_text"].dropna()
)

overlap = train_reviews.intersection(validation_reviews)

print("Unique training reviews:", len(train_reviews))
print("Unique validation reviews:", len(validation_reviews))
print("Overlapping reviews:", len(overlap))

Unique training reviews: 17717
Unique validation reviews: 1779
Overlapping reviews: 0


In [31]:
# Check valid sentiment labels

valid_sentiments = {
    "positive",
    "neutral",
    "negative"
}

invalid_train = train_sample[
    ~train_sample["aspect_sentiment"].isin(valid_sentiments)
]

invalid_validation = validation_sample[
    ~validation_sample["aspect_sentiment"].isin(valid_sentiments)
]

print("Invalid training labels:", len(invalid_train))
print("Invalid validation labels:", len(invalid_validation))

Invalid training labels: 0
Invalid validation labels: 0


In [32]:
# Validate instruction dataset format

def validate_instruction_data(data):

    errors = []

    for i, sample in enumerate(data):

        if not all(
            key in sample
            for key in ["instruction", "input", "output"]
        ):
            errors.append(i)
            continue

        try:
            output = json.loads(sample["output"])

            if output.get("sentiment") not in {
                "positive",
                "neutral",
                "negative"
            }:
                errors.append(i)

        except (json.JSONDecodeError, TypeError):
            errors.append(i)

    return errors


train_errors = validate_instruction_data(training_data)
validation_errors = validate_instruction_data(validation_data)

print("Training format errors:", len(train_errors))
print("Validation format errors:", len(validation_errors))

Training format errors: 0
Validation format errors: 0


In [33]:
# Check training and validation file sizes

train_size = output_path.stat().st_size / (1024 * 1024)

validation_size = (
    validation_output_path.stat().st_size / (1024 * 1024)
)

print(f"Training file size: {train_size:.2f} MB")
print(f"Validation file size: {validation_size:.2f} MB")

Training file size: 9.20 MB
Validation file size: 0.93 MB


## Conclusion

This notebook prepared the training and validation datasets for LoRA fine-tuning in the Heard project.

Key Results:

* 18,000 balanced training samples
* 1,800 balanced validation samples
* Six aspects and three sentiment classes
* No exact review overlap between training and validation samples
* No invalid sentiment labels or instruction formatting errors

Limitations:

Sentiment labels are derived from aspect ratings rather than manual text annotations. Therefore, some labels may not fully reflect the sentiment expressed in the review text.

The prepared datasets support aspect-conditioned sentiment classification but do not directly train the model to identify which aspects are mentioned in a review.

Next Step: B2 — LoRA Fine-tuning

The generated JSONL files will be used to fine-tune a small instruction-following language model and evaluate its performance on aspect-level sentiment classification.